In [33]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense
from tensorflow.keras.utils import to_categorical
import pandas as pd



In [34]:
# Load dataset
df = pd.read_csv("mitbih_train.csv", header=None)

In [35]:
df.head()


,0,1,2,3,4,5,6,7,8,9,...,178,179,180,181,182,183,184,185,186,187
0,0.977941,0.926471,0.681373,0.245098,0.154412,0.191176,0.151961,0.085784,0.058824,0.049020,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.960114,0.863248,0.461538,0.196581,0.094017,0.125356,0.099715,0.088319,0.074074,0.082621,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,1.000000,0.659459,0.186486,0.070270,0.070270,0.059459,0.056757,0.043243,0.054054,0.045946,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.925414,0.665746,0.541436,0.276243,0.196133,0.077348,0.071823,0.060773,0.066298,0.058011,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.967136,1.000000,0.830986,0.586854,0.356808,0.248826,0.145540,0.089202,0.117371,0.150235,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:


# Split features and labels
X = df.iloc[:, :-1].values
y = df.iloc[:, -1].values

# Reshape for CNN
X = X.reshape(X.shape[0], X.shape[1], 1)

# One-hot encoding
y = to_categorical(y)

# Build model
model = Sequential([
    Conv1D(32, 3, activation='relu', input_shape=(X.shape[1], 1)),
    MaxPooling1D(2),

    Conv1D(64, 3, activation='relu'),
    MaxPooling1D(2),

    Flatten(),
    Dense(64, activation='relu'),
    Dense(5, activation='softmax')  # 5 classes
])

model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

# Train
model.fit(X, y, epochs=5, batch_size=32)

# Save model
model.save("ecg_model.h5")

print("✅ Model trained and saved!")

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/5
 323/2737 ━━━━━━━━━━━━━━━━━━━━ 33s 14ms/step - accuracy: 0.8464 - loss: 0.6039

In [ ]:
from tensorflow.keras.models import load_model
import pandas as pd
from tensorflow.keras.utils import to_categorical

# Load the saved model (if not already loaded)
model = load_model('ecg_model.h5')

# Load the test dataset (if not already loaded)
df_test = pd.read_csv("mitbih_test.csv", header=None)

# Split features and labels for test data
X_test = df_test.iloc[:, :-1].values
y_test = df_test.iloc[:, -1].values

# Reshape for CNN (test data)
X_test = X_test.reshape(X_test.shape[0], X_test.shape[1], 1)

# One-hot encoding (test data)
y_test = to_categorical(y_test)

# Evaluate the model on the entire test set
loss, accuracy = model.evaluate(X_test, y_test, verbose=1)

print(f"\nModel Accuracy on Test Set: {accuracy * 100:.2f}%")
print(f"Model Loss on Test Set: {loss:.4f}")

In [ ]:
# --- USER INPUT SECTION ---

# Replace this array with your 187-element ECG signal data.
# For demonstration purposes, I'm using an existing sample from the test set.
user_ecg_signal_raw = X_test[123, :, 0] # Example: Using an arbitrary sample from the test set

# OPTIONAL: If you know the true class of your input, specify its index (0-4).
# This will help in checking if the prediction is correct.
# 0: Normal
# 1: Supraventricular ectopic beat
# 2: Ventricular ectopic beat
# 3: Fusion beat
# 4: Unclassifiable beat
user_true_label_index = y_test[123].argmax() # Example: Get true label for the chosen test sample

print("✅ User ECG signal data defined.")

Now, let's preprocess your input, make a prediction, and generate the analysis report.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# -------------------------------
# ✅ STEP 1: LOAD USER ECG DATA
# -------------------------------

# Option 1: Load from CSV file (RECOMMENDED)
df = pd.read_csv("ventricular_ecg.csv", header=None)
user_ecg_signal_raw = df.values.flatten()

# Option 2: Manual input (uncomment if needed)
# user_input = input("Enter ECG values separated by comma: ")
# user_ecg_signal_raw = np.array([float(x) for x in user_input.split(",")])

# -------------------------------
# ✅ FIX INPUT LENGTH (IMPORTANT)
# -------------------------------
user_ecg_signal_raw = np.array(user_ecg_signal_raw).flatten()

# Convert any length → 187
if len(user_ecg_signal_raw) != 187:
    user_ecg_signal_raw = np.interp(
        np.linspace(0, len(user_ecg_signal_raw) - 1, 187),
        np.arange(len(user_ecg_signal_raw)),
        user_ecg_signal_raw
    )

# -------------------------------
# ✅ STEP 2: STANDARDIZATION (Zero Mean, Unit Variance)
# -------------------------------
# Calculate mean and standard deviation
mean = np.mean(user_ecg_signal_raw)
std_dev = np.std(user_ecg_signal_raw)

# Apply standardization: (data - mean) / std_dev
# Avoid division by zero if std_dev is very small
if std_dev > 1e-7:
    user_ecg_signal_raw = (user_ecg_signal_raw - mean) / std_dev
else:
    user_ecg_signal_raw = user_ecg_signal_raw - mean # Centering only if std_dev is zero


# ✅ STEP 3: RESHAPE FOR MODEL

user_signal_for_prediction = user_ecg_signal_raw.reshape(1, 187, 1)


# ✅ STEP 4: PREDICTION

user_prediction_probabilities = model.predict(user_signal_for_prediction, verbose=0)

predicted_class_index = np.argmax(user_prediction_probabilities)
predicted_probability = np.max(user_prediction_probabilities)

# Class labels
class_labels = [
    'Normal',
    'Supraventricular ectopic beat',
    'Ventricular ectopic beat',
    'Fusion beat',
    'Unclassifiable beat'
]

predicted_class_name = class_labels[predicted_class_index]


# ✅ STEP 5: REPORT

print("\n--- ECG Signal Analysis Report ---")
print(f"Predicted Class: {predicted_class_name}")
print(f"Confidence: {predicted_probability * 100:.2f}%")

print("\nAll Class Probabilities:")
for i, prob in enumerate(user_prediction_probabilities[0]):
    print(f"{class_labels[i]}: {prob:.4f}")

# ✅ STEP 6: VISUALIZATION
prob_df = pd.DataFrame({
    'Class': class_labels,
    'Probability': user_prediction_probabilities[0]
})

plt.figure(figsize=(10, 6))
sns.barplot(x='Class', y='Probability', data=prob_df)

plt.title('Prediction Probabilities')
plt.xlabel('ECG Class')
plt.ylabel('Probability')
plt.xticks(rotation=45)
plt.ylim(0, 1)

plt.show()